# 07 — Results, Excel Report, and Visual Outputs

This notebook aggregates all experiment outputs, builds the Excel summary workbook, and creates final figures.

In [ ]:
from pathlib import Path
import os

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "config" / "genes.json").exists():
            return candidate
    raise FileNotFoundError("Could not find project root. Start Jupyter from repo root or notebooks folder.")

PROJECT_ROOT = find_project_root()
shared_notebook = PROJECT_ROOT / "notebooks" / "_Shared_Training_Utilities.ipynb"
get_ipython().run_line_magic("run", str(shared_notebook))

In [ ]:
metrics = aggregate_all_metrics()
histories = aggregate_all_histories()
summary = summarize_experiments(metrics)

print("Metrics shape:", metrics.shape)
print("Histories shape:", histories.shape)
display(summary)

In [ ]:
# Per-target metric table
if not metrics.empty:
    per_target = metrics.sort_values(["target", "r2"], ascending=[True, False])
    display(per_target.head(20))
else:
    per_target = pd.DataFrame()

In [ ]:
# Best experiment by target
if not metrics.empty:
    idx = metrics.groupby("target")["r2"].idxmax()
    best_by_target = metrics.loc[idx].sort_values("target")
    display(best_by_target[["target", "experiment_name", "model_family", "r2", "mae", "rmse"]])
else:
    best_by_target = pd.DataFrame()

## Visual: mean R² by experiment

In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df = summary.sort_values("mean_r2", ascending=True)
    ax.barh(plot_df["experiment_name"], plot_df["mean_r2"])
    ax.set_title("Mean R² by experiment")
    ax.set_xlabel("Mean R² across TAA targets")
    plt.tight_layout()
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / "mean_r2_by_experiment.png", dpi=300, bbox_inches="tight")
    plt.show()

## Visual: per-target R² by experiment

In [ ]:
if not metrics.empty:
    r2_pivot = metrics.pivot_table(index="target", columns="experiment_name", values="r2", aggfunc="mean").sort_index()

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(r2_pivot.index))
    width = 0.8 / max(len(r2_pivot.columns), 1)

    for i, experiment_name in enumerate(r2_pivot.columns):
        ax.bar(x + i * width, r2_pivot[experiment_name], width=width, label=experiment_name)

    ax.set_xticks(x + width * (len(r2_pivot.columns) - 1) / 2)
    ax.set_xticklabels(r2_pivot.index, rotation=45, ha="right")
    ax.set_ylabel("R²")
    ax.set_title("Per-target R² by experiment")
    ax.axhline(0, linewidth=1)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "per_target_r2_by_experiment.png", dpi=300, bbox_inches="tight")
    plt.show()

## Visual: training curves

In [ ]:
if not histories.empty:
    fig, ax = plt.subplots(figsize=(10, 5))

    for experiment_name, sub in histories.groupby("experiment_name"):
        ax.plot(sub["epoch"], sub["val_loss"], marker="o", label=experiment_name)

    ax.set_title("Validation loss by experiment")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "validation_loss_by_experiment.png", dpi=300, bbox_inches="tight")
    plt.show()

## Visual: copy number vs RNA expression

In [ ]:
modeling_table_path = PROCESSED_DATA_DIR / "modeling_table.csv"
if modeling_table_path.exists():
    modeling_table = pd.read_csv(modeling_table_path)

    def copy_number_vs_expression(df, gene):
        cn_col = f"{gene}_cn"
        expr_col = f"{gene}_rna_log2tpm"

        if cn_col not in df.columns or expr_col not in df.columns:
            return None

        plot_df = df[[cn_col, expr_col]].copy()
        plot_df[cn_col] = pd.to_numeric(plot_df[cn_col], errors="coerce")
        plot_df[expr_col] = pd.to_numeric(plot_df[expr_col], errors="coerce")
        plot_df = plot_df.dropna()

        corr = plot_df[[cn_col, expr_col]].corr().iloc[0, 1]

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(plot_df[cn_col], plot_df[expr_col], alpha=0.35)
        ax.set_xlabel(f"{gene} copy number")
        ax.set_ylabel(f"{gene} RNA log2(TPM + 1)")
        ax.set_title(f"{gene}: copy number vs RNA expression, r={corr:.3f}")
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / f"{gene}_copy_number_vs_expression.png", dpi=300, bbox_inches="tight")
        plt.show()

        return corr

    erbb2_corr = copy_number_vs_expression(modeling_table, "ERBB2")
    print("ERBB2 copy-number/RNA correlation:", erbb2_corr)
else:
    print("modeling_table.csv not found.")

## Build Excel report

In [ ]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
excel_path = REPORTS_DIR / "tcga_taa_experiment_summary.xlsx"

readme_sheet = pd.DataFrame({
    "field": [
        "project",
        "description",
        "targets",
        "features",
        "metrics",
        "note",
    ],
    "value": [
        "TCGA TAA Deep Learning Pipeline",
        "Neural network architectures for predicting tumor-associated antigen RNA expression.",
        ", ".join(TARGET_COLS),
        "Clinical/cancer type, mutation, and gene-level copy-number features.",
        "R², MAE, RMSE, training/validation loss.",
        "Raw GDC data is not included in the GitHub repository because of size.",
    ],
})

# Load configs as table.
config_rows = []
for config_file in sorted((RESULTS_DIR / "configs").glob("*_config.json")):
    with open(config_file) as f:
        cfg = json.load(f)
    flat = {}
    for k, v in cfg.items():
        if isinstance(v, (list, dict)):
            flat[k] = json.dumps(v)
        else:
            flat[k] = v
    config_rows.append(flat)

configs_df = pd.DataFrame(config_rows)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    readme_sheet.to_excel(writer, sheet_name="README", index=False)
    summary.to_excel(writer, sheet_name="experiment_summary", index=False)
    metrics.to_excel(writer, sheet_name="per_target_metrics", index=False)
    histories.to_excel(writer, sheet_name="training_history", index=False)
    configs_df.to_excel(writer, sheet_name="experiment_configs", index=False)
    best_by_target.to_excel(writer, sheet_name="best_model_by_target", index=False)

print("Excel report saved to:", excel_path)

## Add custom project visuals below

In [ ]:
# Add your own visualization code here
